In [28]:
#Script  dim_customers
#Generates 1,000 rows.
#Key name: cust_id (intentional mismatch vs fact_orders.customer_id).
#Injects: nulls, duplicates, mixed phone/date formats, boolean chaos,
#         inconsistent ID prefixes, outliers.

In [29]:
import os, random
import pandas as pd
import numpy as np
from faker import Faker

In [30]:
fake = Faker()
random.seed(42)
np.random.seed(42)
Faker.seed(42)

In [31]:
os.makedirs("data/raw_1", exist_ok=True)

In [32]:
N=1000
SEGMENTS  = ["B2B", "B2C", "Guest"]
SEG_W     = [0.25, 0.60, 0.15]
STATUSES  = ["Active", "Inactive", "Churned"]
STAT_W    = [0.65, 0.20, 0.15]
REGIONS   = ["North", "South", "East", "West", "Central"]
US_STATES = [
    "CA", "TX", "NY", "FL", "IL", "PA", "OH", "GA",
    "NC", "MI", "WA", "AZ", "CO", "MA", "TN"
]

In [33]:
# Helper: mixed date format

In [34]:
def mixed_date(dt, style):
    ts = pd.Timestamp(dt)
    if style == 0:
        return ts.strftime("%m/%d/%Y")
    elif style == 1:
        return ts.strftime("%Y-%m-%d")
    else:
        return ts.strftime("%d-%b-%Y")

In [35]:
# Helper: mixed phone format

In [36]:
def mixed_phone(style):
    d = [str(random.randint(2, 9))] + [str(random.randint(0,9))for _ in range(9)]
    p = "".join(d)
    if style == 0:
        return f"({p[:3]}) {p[3:6]} - {p[6:]}"
    elif style == 1:
        return f"{p[:3]}-{p[3:6]} - {p[6:]}"
    elif style == 2:
        return f"+1{p}"
    else:
        return p

In [37]:
# Helper: boolean as mixed repr

In [38]:
BOOL_STYLES = ["True", "False", "Yes", "No", "1", "0"]

In [39]:
# Base records

In [40]:
records = []
for i in range(1, N + 1):
    join_date = fake.date_between(start_date="-6y", end_date="today")
    phone_style = random.randint(0, 3)
    date_style = random.randint(0, 2)
    
    records.append({                                  # ← indented INSIDE the loop
        "cust_id":      f"CUST-{str(i).zfill(3)}",
        "full_name":    fake.name(),
        "email":        fake.email(),
        "phone":        mixed_phone(phone_style),
        "region":       random.choice(REGIONS),
        "state":        random.choice(US_STATES),
        "segment":      random.choices(SEGMENTS, SEG_W)[0],
        "join_date":    mixed_date(join_date, date_style),
        "status":       random.choices(STATUSES, STAT_W)[0],
    })

df = pd.DataFrame(records)
print(f"Generated {len(df)} rows")

Generated 1000 rows


In [41]:
# PART 2 - Data quality injections

In [42]:
# a) Nulls in 4 columns

In [43]:
for col, rate in [("phone", 0.12), ("state", 0.10),
                 ("segment", 0.11), ("email", 0.13)]:
    null_idx = df.sample(frac=rate, random_state =42).index
    df.loc[null_idx, col] = np.nan

In [44]:
# b) Near-duplicates: same record, name casing / spacing tweaks

In [45]:
dup_idx = df.sample(frac=0.04, random_state=7).index
dups = df.loc[dup_idx].copy()
dups["full_name"] = dups["full_name"].str.upper     # all caps variant
dups["cust_id"] = dups["cust_id"].str.replace("CUST-0", "CUST-00")   # extar zero
df = pd.concat([df, dups], ignore_index = True)

In [46]:
# c) Incomsistent cust_id prefixes

In [47]:
prefix_idx = df.sample(frac=0.06, random_state=55).index
df.loc[prefix_idx, "cust_id"] = (
    df.loc[prefix_idx, "cust_id"]
      .str.replace("CUST-", "", regex=False)
      .str.lstrip("0")
      .apply(lambda x: f"C{x}")
)

In [48]:
# d) Mixed boolean for "is_active" derived from status

In [49]:
def status_to_bool(s):
    style = random.randint(0, 2)
    active = (s=="Active")
    if style == 0:
        return "True" if active else "False"
    elif style == 1:
        return "YES" if active else "NO"
    else:
        return 1 if active else 0

df["is_active"] = df["status"].apply(status_to_bool)

In [50]:
# e) Outlier: age=999 injected as a synthetic 'customer_age' column

In [51]:
df["customer_age"] = [random.randint(18, 75)for _ in range(len(df))]
outlier_idx = df.sample(frac=0.02, random_state=33).index
df.loc[outlier_idx, "customer_age"] = 999

In [52]:
# f) Shuffle and save

In [53]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
out_path = "D:/Portfolio/Projects/retail_portfolio/data/raw/dim_customers.csv"
df.to_csv(out_path, index=False)
print(f"✅ dim_customers saved → {out_path}  |  shape: {df.shape}")
print(df.dtypes)
print(df.head(3))

✅ dim_customers saved → D:/Portfolio/Projects/retail_portfolio/data/raw/dim_customers.csv  |  shape: (1040, 11)
cust_id         object
full_name       object
email           object
phone           object
region          object
state           object
segment         object
join_date       object
status          object
is_active       object
customer_age     int64
dtype: object
    cust_id        full_name                email         phone   region  \
0  CUST-137   Ronald Spencer                  NaN           NaN  Central   
1  CUST-629  Caroline Bailey  chanson@example.org  +18093141818     West   
2  CUST-185   Jonathan Young                  NaN           NaN  Central   

  state segment    join_date    status is_active  customer_age  
0   NaN     NaN  19-Mar-2025  Inactive     False            30  
1    TX     B2C   2020-08-13    Active       YES            63  
2    TX     B2C   2023-12-10    Active      True            34  
